# 🛡️ Hate Speech and Toxic Comment Detection (English & Hinglish)
### Final-Year Engineering Project Pipeline
**Models:** Classical ML (Dual TF-IDF + Linear SVM) vs. Multilingual Transformers (`google/muril-base-cased` & `distilbert-base-multilingual-cased`)

**Colab Instructions:**
1. Go to **Runtime** → **Change runtime type** → Select **T4 GPU**.
2. Run all cells sequentially (`Ctrl + F9`).

In [ ]:
# 1. Environment & GPU Check
!nvidia-smi
!pip install -q transformers datasets accelerate emoji scikit-learn seaborn matplotlib

**Reasoning**:
We will create the required directory structure, write the `requirements.txt` file, and install all dependencies including 'streamlit' and 'lime' to prepare the environment.



In [1]:
import os

# 1. Create directory structure
os.makedirs('src', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# 2. Write requirements.txt
requirements_content = """transformers
datasets
accelerate
emoji
scikit-learn
seaborne
matplotlib
streamlit
lime
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("Created directories and wrote requirements.txt.")

# 3. Install packages from requirements.txt
!pip install -r requirements.txt


Created directories and wrote requirements.txt.
ERROR: Could not find a version that satisfies the requirement seaborne (from versions: none)
ERROR: No matching distribution found for seaborne


**Reasoning**:
The previous installation failed because of a typo in the package name ('seaborne' instead of 'seaborn'). I will modify the requirements.txt file to use the correct spelling 'seaborn' and run the pip installation again.



In [2]:
import os

# 1. Create directory structure
os.makedirs('src', exist_ok=True)
os.makedirs('models', exist_ok=True)
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# 2. Write correct requirements.txt (fixing seaborne -> seaborn)
requirements_content = """transformers
datasets
accelerate
emoji
scikit-learn
seaborn
matplotlib
streamlit
lime
"""
with open('requirements.txt', 'w') as f:
    f.write(requirements_content.strip())

print("Created directories and wrote corrected requirements.txt.")

# 3. Install packages from corrected requirements.txt
!pip install -r requirements.txt

Created directories and wrote corrected requirements.txt.
  Using cached emoji-2.16.0-py3-none-any.whl.metadata (5.7 kB)
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 610.9/610.9 kB 17.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 100.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.4/11.4 MB 104.3 MB/s  0:00:00
  Created wheel for lime: filename=lime-0.2.0.1-py3-none-any.whl size=283913 sha256=029c851aad84c1fa2357cc85bae111d25f574227f5b0133eb52b82462c3223f9
  Stored in directory: /root/.cache/pip/wheels/7c/04/5c/157dc9106512a6c7a30653ec064490c94a49e0fc8f63d19ab9
Successfully built lime
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4/4 [streamlit]


## 2. Text Preprocessing & Hinglish Normalization Engine

In [ ]:
import os, re, urllib.request
import numpy as np, pandas as pd
import emoji

LABEL_TO_ID = {'HATE': 0, 'OFFENSIVE': 1, 'NORMAL': 2}
ID_TO_LABEL = {0: 'HATE', 1: 'OFFENSIVE', 2: 'NORMAL'}
TARGET_NAMES = ['HATE', 'OFFENSIVE', 'NORMAL']

HINGLISH_MAP = {
    'kyaa': 'kya', 'kia': 'kya', 'kyun': 'kyun', 'nahin': 'nahi', 'nai': 'nahi', 'nhn': 'nahi', 'ni': 'nahi',
    'haan': 'ha', 'haa': 'ha', 'hai': 'hai', 'he': 'hai', 'hain': 'hai', 'raha': 'raha', 'rha': 'raha',
    'karna': 'karna', 'krna': 'karna', 'bhai': 'bhai', 'bhaiya': 'bhai', 'bro': 'bhai', 'yaar': 'yaar',
    'kutta': 'kutte', 'kuttey': 'kutte', 'kamina': 'kamine', 'saala': 'saale', 'sala': 'saale',
    'chutiya': 'chutiya', 'chutiye': 'chutiya', 'madarchod': 'madarchod', 'mc': 'madarchod',
    'bhenchod': 'bhenchod', 'bc': 'bhenchod', 'bhosdike': 'bhosdike', 'bsdk': 'bhosdike', 'gandu': 'gandu'
}

def clean_text(text: str) -> str:
    if not isinstance(text, str) or not text.strip():
        return ''
    text = text.lower()
    text = emoji.demojize(text, delimiters=(' ', ' ')).replace(':', ' ')
    text = re.sub(r'https?://\S+|www\.\S+', ' ', text)
    text = re.sub(r'@\w+', ' ', text)
    text = re.sub(r'\brt\b', ' ', text)
    text = re.sub(r'(.)\1{2,}', r'\1\1', text)
    text = re.sub(r'[^\w\s]', ' ', text)
    tokens = [HINGLISH_MAP.get(tok, tok) for tok in text.split()]
    return re.sub(r'\s+', ' ', ' '.join(tokens)).strip()

print('Sample clean test:', clean_text('Abe saalaaa kutteeee https://test.com @user kyaaa haaal hai 😡'))

## 3. Dataset Download, Unified Schema & Stratified 80/10/10 Splits

In [ ]:
from sklearn.model_selection import train_test_split
os.makedirs('data/raw', exist_ok=True)
os.makedirs('data/processed', exist_ok=True)

# Download Davidson Dataset
davidson_url = 'https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv'
urllib.request.urlretrieve(davidson_url, 'data/raw/davidson_raw.csv')
df_en_raw = pd.read_csv('data/raw/davidson_raw.csv')
df_en = pd.DataFrame({'text': df_en_raw['tweet'], 'label': df_en_raw['class'].astype(int), 'language': 'english'})

# Balance English subset (1000 per class)
sampled_en = [group.sample(n=min(len(group), 1000), random_state=42) for _, group in df_en.groupby('label')]
df_en = pd.concat(sampled_en, ignore_index=True)
df_en['clean_text'] = df_en['text'].apply(clean_text)
df_en = df_en[df_en['clean_text'].str.len() > 2]

# Hinglish Code-Mixed Dataset
hinglish_samples = [
    ('in sab ghuspaithiye logo ko desh se bahar nikalo warna ye sab khatam kar denge', 0),
    ('ye log desh ke gaddar hai inko goli maaro saalo ko', 0),
    ('all people of this religion are terrorists and deserve no human rights', 0),
    ('in minority walo ko sabak sikhana padega ye log virus ki tarah fail rahe hai', 0),
    ('inke religious places ko tod dalo saale jihadi log', 0),
    ('abe saale chutiye apna kaam kar zyada gyan mat pel', 1),
    ('tu bilkul gadha hai dimag ghutne me hai kya tera', 1),
    ('teri aukaat nahi hai mujhse baat karne ki chup baith bhenchod', 1),
    ('what a madarchod referee cheating karwa raha tha match me', 1),
    ('bhosdike ke chup kar warna thappad marunga', 1),
    ('bhai kal cricket match dekha kya kya zabardast six mara tha virat ne', 2),
    ('aaj ka mausam bohot accha hai chalo chai peene chalte hai', 2),
    ('bhai tu kaisa hai bohot time baad message kiya tune', 2),
    ('weekend pe goa chalne ka plan banate hai sab milke', 2),
    ('diwali ki shopping ke liye market me bahut mast lights lagi hai', 2)
] * 80

df_hi = pd.DataFrame({'text': [s[0] for s in hinglish_samples], 'label': [s[1] for s in hinglish_samples], 'language': 'hinglish'})
df_hi['clean_text'] = df_hi['text'].apply(clean_text)

# Stratified 80/10/10 Splits
tr_en, tmp_en = train_test_split(df_en, test_size=0.2, random_state=42, stratify=df_en['label'])
val_en, te_en = train_test_split(tmp_en, test_size=0.5, random_state=42, stratify=tmp_en['label'])

tr_hi, tmp_hi = train_test_split(df_hi, test_size=0.2, random_state=42, stratify=df_hi['label'])
val_hi, te_hi = train_test_split(tmp_hi, test_size=0.5, random_state=42, stratify=tmp_hi['label'])

train_df = pd.concat([tr_en, tr_hi]).sample(frac=1.0, random_state=42).reset_index(drop=True)
val_df = pd.concat([val_en, val_hi]).sample(frac=1.0, random_state=42).reset_index(drop=True)
test_comb = pd.concat([te_en, te_hi]).sample(frac=1.0, random_state=42).reset_index(drop=True)

print(f'Train count: {len(train_df)}, Val count: {len(val_df)}, Test Combined: {len(test_comb)}')

## 4. Classical ML Baselines (TF-IDF + Linear SVM & Logistic Regression)

In [ ]:
from sklearn.pipeline import Pipeline, FeatureUnion
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.calibration import CalibratedClassifierCV
from sklearn.svm import LinearSVC
from sklearn.metrics import classification_report, f1_score

features = FeatureUnion([
    ('word', TfidfVectorizer(ngram_range=(1, 2), max_features=10000, sublinear_tf=True)),
    ('char', TfidfVectorizer(ngram_range=(3, 5), max_features=15000, analyzer='char', sublinear_tf=True))
])

svm_pipeline = Pipeline([
    ('features', features),
    ('clf', CalibratedClassifierCV(LinearSVC(class_weight='balanced', random_state=42), cv=3))
])

svm_pipeline.fit(train_df['clean_text'], train_df['label'])
preds = svm_pipeline.predict(test_comb['clean_text'])
print('--- Linear SVM (Combined Test Set) ---')
print(classification_report(test_comb['label'], preds, target_names=TARGET_NAMES, digits=4))

## 5. Fine-Tuning Transformer: `google/muril-base-cased` with Class Weights

In [ ]:
import torch, torch.nn as nn
from torch.utils.data import Dataset
from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments, EarlyStoppingCallback
from sklearn.utils.class_weight import compute_class_weight

MODEL_NAME = 'google/muril-base-cased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

class ToxDataset(Dataset):
    def __init__(self, texts, labels):
        self.encodings = tokenizer(texts.tolist(), truncation=True, padding=True, max_length=128, return_tensors='pt')
        self.labels = torch.tensor(labels.tolist(), dtype=torch.long)
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, idx):
        item = {k: v[idx] for k, v in self.encodings.items()}
        item['labels'] = self.labels[idx]
        return item

train_ds = ToxDataset(train_df['clean_text'], train_df['label'])
val_ds = ToxDataset(val_df['clean_text'], val_df['label'])
test_ds = ToxDataset(test_comb['clean_text'], test_comb['label'])

# Class weights
weights = compute_class_weight('balanced', classes=np.array([0, 1, 2]), y=train_df['label'].values)
weights_tensor = torch.tensor(weights, dtype=torch.float).cuda() if torch.cuda.is_available() else torch.tensor(weights, dtype=torch.float)

model = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=3, id2label=ID_TO_LABEL, label2id=LABEL_TO_ID)

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False):
        labels = inputs.get('labels')
        outputs = model(**inputs)
        logits = outputs.get('logits')
        loss_fn = nn.CrossEntropyLoss(weight=weights_tensor.to(model.device))
        loss = loss_fn(logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    f1 = f1_score(labels, preds, average='macro')
    return {'macro_f1': f1}

training_args = TrainingArguments(
    output_dir='./muril_checkpoints',
    num_train_epochs=3,
    per_device_train_batch_size=16,
    learning_rate=2e-5,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='macro_f1',
    fp16=torch.cuda.is_available(),
    report_to='none'
)

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)]
)

trainer.train()
trainer.save_model('./muril_final_model')
tokenizer.save_pretrained('./muril_final_model')

## 6. Confusion Matrix & Cross-Lingual Evaluation

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix

muril_preds_raw = trainer.predict(test_ds)
muril_preds = np.argmax(muril_preds_raw.predictions, axis=1)

cm = confusion_matrix(test_comb['label'], muril_preds, labels=[0, 1, 2])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', xticklabels=TARGET_NAMES, yticklabels=TARGET_NAMES)
plt.title('Confusion Matrix: MuRIL (Combined Test Set)')
plt.xlabel('Predicted')
plt.ylabel('Actual')
plt.show()

print('--- Final MuRIL Benchmark ---')
print(classification_report(test_comb['label'], muril_preds, target_names=TARGET_NAMES, digits=4))

# Task
```xml
<task>
Build and evaluate a comparative NLP pipeline for hate speech and toxic comment detection across English and Hinglish (code-mixed) text. Your tasks are:

1. **Environment and Project Structure Setup**:
   Create a clean directory structure with a `requirements.txt` file containing dependencies: `transformers`, `datasets`, `accelerate`, `emoji`, `scikit-learn`, `seaborn`, `matplotlib`, `streamlit`, and `lime`.

2. **Data Pipeline (`data_prep.py`)**:
   Implement a dataset downloader and text preprocessing module.
   - Download the Davidson dataset from "https://raw.githubusercontent.com/t-davidson/hate-speech-and-offensive-language/master/data/labeled_data.csv" to serve as the English corpus.
   - Programmatically assemble the synthetic Hinglish (code-mixed) dataset.
   - Create a robust normalization engine (`clean_text`) featuring emoji expansion, URL/handle removal, character repetition normalization, and a phonetic dictionary mapping to resolve variation in common Hinglish terms.
   - Create balanced stratified training, validation, and combined testing datasets (80/10/10 splits) and save them to the disk.

3. **Baseline Pipeline (`train_baseline.py`)**:
   - Fit a feature extraction pipeline utilizing a union of word-level and character-level TF-IDF vectorizers.
   - Train and calibrate a Linear Support Vector Machine (Linear SVM) and a Logistic Regression baseline model using class balancing.
   - Output baseline evaluation performance statistics.

4. **Transformer Training Pipeline (`train_transformer.py`)**:
   - Write a fine-tuning script utilizing PyTorch and Hugging Face's `Trainer` API to train the multilingual transformer models `google/muril-base-cased` and `distilbert-base-multilingual-cased`.
   - Implement custom training arguments with `EarlyStoppingCallback` and class-weighting within the loss computation to address any class imbalance.
   - Save the best trained model state and tokenizer.

5. **Evaluation, Error Analysis, and Interpretability (`evaluate.py`)**:
   - Compute final classification metrics (Precision, Recall, F1-Score) separately for English and Hinglish subsets.
   - Generate and plot Confusion Matrices using Seaborn.
   - Implement an error analysis module displaying the 20 worst misclassifications.
   - Set up an explainability module using LIME (Local Interpretable Model-agnostic Explanations) to showcase word-level contributions for toxic predictions.

6. **Streamlit Deployment App (`app.py`)**:
   - Build a highly interactive web demonstration interface using Streamlit.
   - Allow users to choose between the classical baseline model and the fine-tuned transformer model.
   - Display predicted class labels, confidence scores, and highlight significant words contributing to the prediction.
</task>
```

## Project Structure & Requirements Setup

### Subtask:
Create the project directory structure and write the `requirements.txt` file listing all necessary dependencies.
